# <span> Import Raw Data </span>

## <span> Find Files </span>

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import sys

# Establish the downloaded AMISOR database as the root from which to find MicroCAT data
root = Path("/g/data/gv90/ad6790/amery-amisor//data/raw/AMISOR")

# Find all MicroCAT data files 
dat_files = sorted(root.rglob("*.dat"))

# Regex (regular expression): find AM01..AM06 anywhere in path/name + cat#### filename
rx = re.compile(r"(AM\d{2}).*?\bcat(\d+)(?:_[^/]*)?\.dat\b", re.IGNORECASE)

## <span> Parse Correct MicroCAT Files to Dataframe Objects </span>

### <span> Parsing Function </span>

<span> Parses in the following values from each .dat file, saving them as the corresponding variable: </span> \
<span> t: in-situ temperature, degrees C </span> \
<span> C: conductivity, mS/cm </span> \
<span> p: pressure, dbar </span> \
<span> SP: practical salinity, PSS78 </span>

In [2]:
# Define a function for parsing MicroCAT data to pandas dataframe object
def parse_microcat_dat(path: Path, skiprows: int = 5) -> pd.DataFrame:
    # Load the file containing data
    data = np.loadtxt(path, skiprows=skiprows)

    # There are several columns, but we are interested in parsing in values related to in situ temperature, conductivity, pressure, practical salinity, and time 
    t = data[:, 1]
    C = data[:, 2]
    p = data[:, 3]
    SP  = data[:, 4]

    day    = data[:, 5].astype(int)
    month  = data[:, 6].astype(int)
    year   = data[:, 7].astype(int)
    hour   = data[:, 8].astype(int)
    minute = data[:, 9].astype(int)
    second = data[:,10].astype(int)

    # Convert parsed datetime data into a standardised pandas datatime object
    time = pd.to_datetime(
        dict(year=year, month=month, day=day, hour=hour, minute=minute, second=second),
        errors="coerce"
    )

    # Store all parsed data into a dataframe object, using the datetime object as the index
    df = pd.DataFrame({"in_situ_temperature": t, "conductivity": C, "pressure": p, "practical_salinity": SP}, index=time).sort_index()
    df.index.name = "time"

    # Replace all null data (~-999) with NaN
    df = df.mask(np.isclose(df, -999.0, atol=1e-6))
    
    # OPTIONAL: Delete lines of data where datetime cannot be parsed properly
    # df = df[~df.index.isna()]
    
    return df


### <span> Ensure each .dat file is assigned to a dataframe object corresponding to a unique borehole location </span>

<span> defaultdict(list) creates a dictionary where any new/missing key is automatically assigned to a new, empty list
i.e. the below code iterates over all the recognised .dat files, parsing the MicroCAT data to a unique dataframe object, and then adding each dataframe object to a list
of dataframe objects identified by a key corresponding to the AM and cat number </span>

<span> It will look something like below:
dfs_list = {
    ("AM01","1969"): [df_part1, df_part2],
    ("AM01","1970"): [df_part1],
    ("AM02","1174"): [df_part1, df_part2, df_part3],
} </span>

<span> In some instances, there are multiple .dat files for a single MicroCAT, often representing different seasonal data downloads. It is necessary to read in all files though, as while there is overlap, each contains unique timestamped data. </span>

In [3]:
# Since there can be multiple .dat files for a single cat (some observational data downloads from different years have been added to the database in separate files)
# Store dataframe objects in a list, keyed by the borehole and cat id (e.g. AM01, cat0001)
dfs_list = defaultdict(list)          # keyed by (AMxx, cat####)
skipped = []                          # files that don’t match the pattern
failed = []                           # files that match but can’t be parsed

# Iterate over all recognised .dat files
for p in dat_files:
    # Skip files containing certain words
    exclude_words = ["uncorrected"]
    name_lower = p.name.lower()
    if any(word in name_lower for word in exclude_words):
        continue
    
    # Use regex to identify borehole (AMXX) and cat#### in .dat file
    m = rx.search(str(p))
    if not m:
        skipped.append(p)
        continue

    # Determine the borehole id and cat id from the filename
    am_id = m.group(1).upper()
    cat_id = m.group(2)  # keep as string; cast to int if you prefer

    # Parse the MicroDAT data and assign to a dataframe object
    # Also assign the borehole, cat id, and source filename to the dataframe
    try:
        df = parse_microcat_dat(p, skiprows=5)
        df["AM"] = am_id
        df["cat"] = int(cat_id)
        df["source_file"] = p.name

        # Use the borehole and cat ids as keys to identify the list of dataframe objects
        dfs_list[(am_id, cat_id)].append(df)
    
    except Exception as e:
        failed.append((p, repr(e)))

print(dfs_list.keys())


dict_keys([('AM03', '4054'), ('AM03', '4055'), ('AM05', '7322'), ('AM05', '7326'), ('AM05', '7327'), ('AM06', '7323'), ('AM06', '7324'), ('AM06', '7325'), ('AM01', '1969'), ('AM01', '1970'), ('AM01', '1971'), ('AM02', '1174'), ('AM02', '1623'), ('AM02', '1624'), ('AM03', '3883'), ('AM04', '1972'), ('AM04', '1973'), ('AM04', '1974')])


### <span> Choose the correct .dat files for cat7327 and cat7325 from the 2011/12 data download </span>

<span> For cat7327 (AM05) and cat7325 (AM06), there are several .dat files in the 2011/12 download. One of these contains "mean" in the filename, and uses the average pressure data from just after deployment. This is due to bad pressure data, and revisions required. Here, these "mean" files need to be selected over the others. </span>

In [4]:
# Iterate over each key (borehole, cat id) in the dict
for key in dfs_list:
    # Define the list of DataFrame object for this iteration of key (e.g. all the DataFrames for AM0X, catXXXX)
    # For each cat, this will correspond to a list of DataFrames from different .dat files
    frames = dfs_list[key]
    # Find the base names of any mean files (e.g. "cat7327_2meanp.dat" -> "cat7327_2.dat")
    mean_bases = set()
    for f in frames:
        name = f["source_file"].iloc[0]
        if "meanp" in name.lower():
            mean_bases.add(name.lower().replace("meanp", ""))

    # Only drop frames whose exact filename is replaced by a mean version
    if mean_bases:
        dfs_list[key] = [
            f for f in frames
            if f["source_file"].iloc[0].lower() not in mean_bases
        ]

## <span> Collate Data into One Dataframe Object </span>

### <span> Function to remove duplicate timestamps </span>

<span> Publicly retrieved data is inconsistently provided as single files of multiple (or all) data downloads, or individual files for one data download. There are therefore overlaps in timestamps between data files for some boreholes. Since multiple dataframe objects may represent a given borehole, identical timestamps need to be resolved to ensure there is no overlap in data. </span>

In [5]:
# Set up a function to resolve different cases for where there are duplicates between data files for a specific timestamp
def resolve_duplicates(df):
    resolved_rows = []
    flagged = []

    # Iterate over each group defined by each timestep => matching timesteps have been grouped together
    for timestamp, group in df.groupby(df.index):

        # If there is a unique timestamp, then there is no duplicate and we are done
        if len(group) == 1:
            resolved_rows.append(group.iloc[0])
            continue

        # Within the duplicate timestamps, we want to only select numerical data types (i.e. t, cond, pres, sal) to compare
        numeric = group.select_dtypes(include=[np.number])

        # Case 1: Data in duplicated timestamps is identical (i.e. there is only one unique set of data values)
        # Arbitrarily take the first row
        if numeric.nunique().max() == 1:
            resolved_rows.append(group.iloc[0])
            continue

        # Case 2: Data in 1+ duplicated timestamps is all -999
        # Identify rows where all numeric columns are -999
        mask_all_bad = numeric.isna().all(axis=1)

        # If at least one good row exists
        if (~mask_all_bad).any():
        
            # Drop all fully-bad rows
            group_clean = group[~mask_all_bad]
            numeric_clean = numeric[~mask_all_bad]
        
            # Case 1: There is remaining duplicated timesteps with identical data
            if numeric_clean.nunique().max() == 1:
                resolved_rows.append(group_clean.iloc[0])
                continue
        
            # Case 3: There is remaining duplicated timesteps with conflicting data
            print("\nConflict at:", timestamp)
            print(group_clean)
            flagged.append((timestamp, group_clean))
        
            resolved_rows.append(group_clean.iloc[0])
            continue

        # Case 3: Different valid values → flag for inspection
        print("\nConflict at:", timestamp)
        print(group)
        flagged.append((timestamp, group))

        # For now, keep first arbitrarily
        resolved_rows.append(group.iloc[0])

    result = pd.DataFrame(resolved_rows)
    result.index.name = "time"

    return result.sort_index(), flagged

### <span> Concatenate dataframes for each borehole into one dataframe </span>

In [6]:
# Set up a dataframe object in which to concatenate the parsed dataframe objects corresponding to each AM and cat
dfs = {}    # keyed by (AMxx, cat####)
    
# Iterate over list of dataframes (pertaining to individual .dat files) defined by each key in the dictionary, combining each list of dataframes into one dataframe
for key, parts in dfs_list.items():
    # Stack all dataframes in the list corresponding to a key and sort by time => then concatenate into one dataframe
    out = pd.concat(parts).sort_index()

    # Since there is minor overlap in data timestamps between files, we choose to keep the last occurrence here
    out, flagged_conflicts = resolve_duplicates(out)

    dfs[key] = out

print(dfs.keys())

print(f"Matched+parsed: {len(dfs)} | skipped (no AM/cat): {len(skipped)} | failed parse: {len(failed)}")

dict_keys([('AM03', '4054'), ('AM03', '4055'), ('AM05', '7322'), ('AM05', '7326'), ('AM05', '7327'), ('AM06', '7323'), ('AM06', '7324'), ('AM06', '7325'), ('AM01', '1969'), ('AM01', '1970'), ('AM01', '1971'), ('AM02', '1174'), ('AM02', '1623'), ('AM02', '1624'), ('AM03', '3883'), ('AM04', '1972'), ('AM04', '1973'), ('AM04', '1974')])
Matched+parsed: 18 | skipped (no AM/cat): 34 | failed parse: 0


### <span> Combine all data into one dataframe object </span>

In [7]:
# Combine into one DataFrame
frames = []
for (borehole, cat), df in dfs.items():
    df = df.reset_index()  # make time a column again
    df["borehole"] = borehole
    df["cat"] = cat
    frames.append(df)

df_all = pd.concat(frames, ignore_index=True)

## <span> Add Coordinate Locations for Each Borehole </span>
<span> To later calculate absolute salinity using practical salinity, the latitude and longitude of the ocean mooring is required. The GPS reading of each borehole was typically taken each summer (or at least estimated from velocity calculations) and reported in each seasonal data processing and quality report for the AMISOR project. These reports were fed into an AI model (Claude - Opus 4.6) to generate a csv file of longitudes and latitudes, which has been verified as correct. This data will be parsed in here and added to each line of the dataframe corresponding to a particular borehole ("AM..") and timestamp. </span> \
\
<span> Each GPS reading is treated as valid from its reading date until the next GPS reading for that borehole. The LAST reading for each borehole is applied to all subsequent data (i.e. extrapolated forward). </span> \
\
<span> The FIRST reading is also back-filled to any data that predates the earliest GPS fix for that borehole (relevant for AM03 whose first position is from Jan 2008 but data starts Dec 2005, and AM05/AM06 whose data starts ~1 year before the first GPS reading). </span> \
\
<span> Positions flagged as 'estimated' in the CSV (derived from ice-shelf velocity calculations rather than direct GPS) are marked in an 'estimated_position' column. </span> 

### <span> Parsing Function </span>


In [8]:
def add_latlon(df: pd.DataFrame, gps_csv: str | Path) -> pd.DataFrame:
    # Load GPS position lookup table
    gps = pd.read_csv(gps_csv)
    # Convert date columns to datetime
    gps["gps_date"] = pd.to_datetime(gps["gps_date"], dayfirst=True)
    gps["valid_from"] = pd.to_datetime(gps["valid_from"], dayfirst=True)
    gps["valid_until"] = pd.to_datetime(gps["valid_until"], dayfirst=True)
    # Sort GPS data by borehole and valid_from date for easier processing
    gps = gps.sort_values(["borehole", "valid_from"]).reset_index(drop=True)

    # Make a copy of the input dataframe to avoid modifying the original
    df = df.copy()
 
    # Initialise output columns
    df["latitude"] = np.nan
    df["longitude"] = np.nan
 
    # Iterate over each borehole in the data
    for bh in df["borehole"].unique():
        # Filters the GPS lookup table to a subset of only current iteration of the borehole
        bh_gps = gps[gps["borehole"] == bh]

        # True/False mask so that we only modify the rows of the dataframe corresponding to the current borehole
        bh_mask = df["borehole"] == bh
 
        # Iterate over each GPS reading for this borehole
        # _ is a throwaway variable for the row index, which we don't need
        for _, row in bh_gps.iterrows():
            # Get the date range for which this GPS reading applies to
            vfrom = row["valid_from"]
            vuntil = row["valid_until"]
 
            # Create a mask that is True for this borehole and the time period this GPS reading applies to
            # pd.isna() checks for empty values
            # Catches the last GPS reading for each borehole, which has no valid_until date
            if pd.isna(vuntil):
                period_mask = bh_mask & (df["time"] >= vfrom)
            else:
                period_mask = bh_mask & (df["time"] >= vfrom) & (df["time"] < vuntil)
 
            # Assign the lat/lon values for this GPS reading to the rows of the dataframe that match the borehole and time period
            df.loc[period_mask, "latitude"] = row["latitude"]
            df.loc[period_mask, "longitude"] = row["longitude"]
 
        # Find the earliest GPS reading data for the current iteration of the borehole
        first_gps_date = bh_gps["valid_from"].min()
        # Find the data rows (if any) that fall before the first GPS reading
        backfill_mask = bh_mask & (df["time"] < first_gps_date)
        # Conditional statement to backfill any timestamps before the first GPS reading
        if backfill_mask.any():
            # Takes the earliest GPS reading for this borehole
            first_row = bh_gps.iloc[0]
            # Assigns this earliest GPS reading to any data rows that fall before the first GPS reading
            df.loc[backfill_mask, "latitude"] = first_row["latitude"]
            df.loc[backfill_mask, "longitude"] = first_row["longitude"]
 
    # Safety check: report any rows that still have no lat/lon assigned after processing all GPS readings
    n_missing = df["latitude"].isna().sum()
    if n_missing > 0:
        missing_bhs = df.loc[df["latitude"].isna(), "borehole"].unique()
        print(f"Warning: {n_missing} rows have no lat/lon assigned. "
              f"Boreholes affected: {list(missing_bhs)}")
 
    return df
 
df_final = add_latlon(df_all, gps_csv="/g/data/gv90/ad6790/amery-amisor/data/raw/amisor_borehole_gps_positions.csv")

### <span> Save Dataframe as Parquet </span>

In [9]:
# Save to parquet file for efficient storage and future use
# Note: parquet is a columnar storage format that is more efficient for large datasets and preserves data types better than CSV
# Note: parquet files can be read back into pandas with pd.read_parquet("data/proc/01_imported.parquet")
# Note: this will overwrite any existing file at this location
df_final.to_parquet("/g/data/gv90/ad6790/amery-amisor/data/proc/01_imported.parquet", index=False)

### <span> Check column names of dataframe </span>

In [10]:
df_final.columns  

Index(['time', 'in_situ_temperature', 'conductivity', 'pressure',
       'practical_salinity', 'AM', 'cat', 'source_file', 'borehole',
       'latitude', 'longitude'],
      dtype='object')

In [11]:
print(df_final.head())

                 time  in_situ_temperature  conductivity  pressure  \
0 2005-12-27 00:00:01              -2.0632       27.2829   872.483   
1 2005-12-27 00:30:01              -2.0575       27.2885   872.483   
2 2005-12-27 01:00:01              -2.0558       27.2912   872.483   
3 2005-12-27 01:30:00              -2.0562       27.2912   872.500   
4 2005-12-27 02:00:00              -2.0565       27.2904   872.483   

   practical_salinity    AM   cat      source_file borehole   latitude  \
0             34.4491  AM03  4054  cat4054_all.dat     AM03 -70.561167   
1             34.4503  AM03  4054  cat4054_all.dat     AM03 -70.561167   
2             34.4521  AM03  4054  cat4054_all.dat     AM03 -70.561167   
3             34.4526  AM03  4054  cat4054_all.dat     AM03 -70.561167   
4             34.4518  AM03  4054  cat4054_all.dat     AM03 -70.561167   

   longitude  
0  70.332167  
1  70.332167  
2  70.332167  
3  70.332167  
4  70.332167  
